# Credit Card Fraud Detection using Machine Learning

**Student:** Madhu Jain  
**Project type:** Data Analytics / Machine Learning  

This notebook demonstrates an end-to-end fraud-classification workflow. Download `creditcard.csv` from the dataset source linked in `README.md` and place it in the same folder as this notebook. The dataset contains anonymized transaction features and a `Class` target (0 = legitimate, 1 = fraud).

## 1. Install and import libraries
If needed, install dependencies with `pip install -r requirements.txt`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight
import joblib

RANDOM_STATE = 42

## 2. Load and inspect the dataset
The standard Kaggle dataset is large. Keep the original dataset unchanged and document any filtering or sampling you perform.

In [ ]:
df = pd.read_csv('creditcard.csv')
print('Shape:', df.shape)
display(df.head())
display(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nTarget distribution:')
display(df['Class'].value_counts(dropna=False))
print('\nFraud rate (%):', round(df['Class'].mean() * 100, 4))

## 3. Exploratory data analysis
Fraud is rare in this dataset, so accuracy alone can hide poor fraud detection.

In [ ]:
df['Class'].value_counts().sort_index().plot(kind='bar')
plt.title('Transaction class distribution')
plt.xlabel('Class (0 = legitimate, 1 = fraud)')
plt.ylabel('Number of transactions')
plt.tight_layout()
plt.show()

display(df.groupby('Class')['Amount'].agg(['count', 'mean', 'median', 'max']))

## 4. Data preparation and train/test split
`Time` and `Amount` are scaled. The anonymized `V1`–`V28` features are retained as supplied. The split is stratified to preserve the rare fraud class in both sets.

In [ ]:
if df.isnull().any().any():
    df = df.dropna().copy()

X = df.drop(columns=['Class']).copy()
y = df['Class'].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
scale_cols = [c for c in ['Time', 'Amount'] if c in X_train.columns]
X_train = X_train.copy()
X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print('Training rows:', len(X_train), '| Test rows:', len(X_test))
print('Training class counts:\n', y_train.value_counts())

## 5. Train models
Logistic Regression provides a baseline. Random Forest can capture non-linear patterns. Class weighting helps account for imbalance; it does not eliminate the need to evaluate false negatives and false positives.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision (fraud)': precision_score(y_test, pred, zero_division=0),
        'Recall (fraud)': recall_score(y_test, pred, zero_division=0),
        'F1 (fraud)': f1_score(y_test, pred, zero_division=0)
    })
    print('\n' + name)
    print(classification_report(y_test, pred, zero_division=0))
    print('Confusion matrix:\n', confusion_matrix(y_test, pred))

results_df = pd.DataFrame(results).set_index('Model')
display(results_df)

## 6. Confusion matrix for the selected model
Select the model based on validation/test performance and the practical cost of missed fraud versus false alerts. Do not select on accuracy alone.

In [ ]:
best_name = results_df['F1 (fraud)'].idxmax()
best_model = models[best_name]
best_pred = best_model.predict(X_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=['Legitimate', 'Fraud'], values_format='d'
)
plt.title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()
print('Selected model by fraud F1-score:', best_name)

## 7. Save model artifacts
Save the model, scaler, and feature order so inference can use the same preprocessing. Test saved artifacts before deployment.

In [ ]:
os.makedirs('artifacts', exist_ok=True)
joblib.dump(best_model, 'artifacts/fraud_detection_model.joblib')
joblib.dump(scaler, 'artifacts/standard_scaler.joblib')
joblib.dump(list(X.columns), 'artifacts/feature_columns.joblib')
print('Saved model, scaler, and feature columns to ./artifacts/')

## 8. Conclusion and limitations
- This is a binary classification example for identifying potentially fraudulent transactions.
- Because the target is highly imbalanced, precision, recall, F1-score, and the confusion matrix are important.
- Results depend on the dataset version, cleaning decisions, split, and model settings; report the results generated by your own run.
- This educational model is not a production fraud-control system. Real deployment needs monitoring, privacy/security controls, cost-sensitive thresholds, drift checks, and human review.